In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report

In [ ]:
import pandas as pd
df=pd.read_csv("test.csv", encoding='latin1')

In [ ]:
df.head()

,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797.0,27400.0,105.0
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265.0,470.0,164.0
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272.0,1246700.0,26.0


In [ ]:
df=df[['textID','text','sentiment']]
df.head()

,textID,text,sentiment
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative
3,01082688c6,happy bday!,positive
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4815 entries, 0 to 4814
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   textID     3534 non-null   object
 1   text       3534 non-null   object
 2   sentiment  3534 non-null   object
dtypes: object(3)
memory usage: 113.0+ KB


In [ ]:
df=df.dropna()

In [ ]:
print(df.isnull().sum())

textID       0
text         0
sentiment    0
dtype: int64


In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    return text

df['cleaned_text'] = df['text'].apply(preprocess_text)
print(df[['text', 'cleaned_text']].head())

                                                text  \
0  Last session of the day  http://twitpic.com/67ezh   
1   Shanghai is also really exciting (precisely -...   
2  Recession hit Veronique Branquinho, she has to...   
3                                        happy bday!   
4             http://twitpic.com/4w75p - I like it!!   

                                        cleaned_text  
0                          last session of the day    
1   shanghai is also really exciting precisely  s...  
2  recession hit veronique branquinho she has to ...  
3                                         happy bday  
4                                          i like it  


In [ ]:

vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['cleaned_text'])
y = df['sentiment'].map({'positive': 1, 'negative': 0, 'neutral': 2})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)


In [ ]:

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['negative', 'positive', 'neutral']))


Accuracy: 60.79%
Classification Report:
              precision    recall  f1-score   support

    negative       0.62      0.53      0.57       306
    positive       0.64      0.72      0.68       332
     neutral       0.57      0.58      0.57       423

    accuracy                           0.61      1061
   macro avg       0.61      0.61      0.61      1061
weighted avg       0.61      0.61      0.61      1061



In [ ]:

def predict_sentiment(review):
    review_cleaned = preprocess_text(review)
    review_vectorized = vectorizer.transform([review_cleaned])
    prediction = nb.predict(review_vectorized)
    sentiment = {0: "negative", 1: "positive", 2: "neutral"}[prediction[0]]
    return sentiment


print(predict_sentiment(""))
print(predict_sentiment("This is the worst experience ever."))


neutral
negative


In [ ]:
def classify_new_input(raw_text):
    # 1. Accept new raw text input
    if not isinstance(raw_text, str) or raw_text.strip() == "":
        return "Invalid input"

    # 2. Preprocess using same function
    cleaned_text = preprocess_text(raw_text)

    # 3. Vectorize using trained vectorizer
    vectorized_text = vectorizer.transform([cleaned_text])

    # 4. Predict class label using trained model
    prediction = nb.predict(vectorized_text)[0]

    # Map numeric label back to sentiment
    label_map = {0: "negative", 1: "positive", 2: "neutral"}

    return label_map[prediction]

In [ ]:
print(classify_new_input("I love this product!"))
print(classify_new_input("This is terrible service."))
print(classify_new_input("It was okay, nothing special."))

positive
negative
positive
